In [ ]:
# Cell 1: Create and use virtualenv inside Colab
from pathlib import Path
import logging
import os
import shutil
import subprocess
import sys

CELL_NAME = "cell_01_setup"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()  # delete old log on rerun

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
REPO_URL = "https://github.com/kareemkamal10/NAMAA-Egyptian-Voice.git"
VENV_DIR = Path("/content/namaa-venv")
VENV_PYTHON = None

logger.info("Start Cell 1")
print(f"Logging to: {LOG_FILE}")

try:
    logger.info("Installing system packages for the isolated Python runtime")
    run_cmd(["apt-get", "update"])
    run_cmd(["apt-get", "install", "-y", "python3.11", "python3.11-venv", "git-lfs"])

    if not PROJECT_DIR.exists():
        logger.info("Cloning repository")
        run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    else:
        logger.info("Repository already exists")

    run_cmd(["git", "lfs", "install"], cwd=str(PROJECT_DIR))
    run_cmd(["git", "lfs", "pull"], cwd=str(PROJECT_DIR))

    preferred_python = shutil.which("python3.11") or shutil.which("python3.10") or sys.executable

    if preferred_python is None:
        raise RuntimeError("No usable Python interpreter found to build the virtualenv.")

    logger.info(f"Using Python for virtualenv: {preferred_python}")
    print(f"Virtualenv Python: {preferred_python}")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "virtualenv"])
    if VENV_DIR.exists():
        logger.info("Removing old virtualenv")
        shutil.rmtree(VENV_DIR)

    subprocess.check_call([sys.executable, "-m", "virtualenv", "--python", preferred_python, str(VENV_DIR)])

    if os.name == "nt":
        VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
    else:
        VENV_PYTHON = VENV_DIR / "bin" / "python"

    logger.info(f"VENV_PYTHON={VENV_PYTHON}")
    print(f"VENV Python: {VENV_PYTHON}")

    run_cmd([str(VENV_PYTHON), "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], cwd=str(PROJECT_DIR))
    run_cmd([str(VENV_PYTHON), "-m", "pip", "install", "-r", "requirements.txt"], cwd=str(PROJECT_DIR))
    logger.info("requirements installed successfully inside virtualenv")
except Exception:
    logger.exception("Cell 1 failed")
    raise

logger.info("End Cell 1")


In [ ]:
# Cell 2: Runtime check inside the virtualenv
from pathlib import Path
import logging
import os
import subprocess

CELL_NAME = "cell_02_runtime_check"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
if os.name == "nt":
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

logger.info("Start Cell 2")
print(f"Logging to: {LOG_FILE}")

try:
    script = """
import sys
import torch

print('Python:', sys.version.replace('\n', ' '))
print('Executable:', sys.executable)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
"""
    run_cmd([str(VENV_PYTHON), "-c", script], cwd=str(PROJECT_DIR))
except Exception:
    logger.exception("Cell 2 failed")
    raise

logger.info("End Cell 2")


In [ ]:
# Cell 3: Prepare inputs inside the virtualenv
from pathlib import Path
import logging
import os
import subprocess

CELL_NAME = "cell_03_prepare_inputs"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
OUTPUT_PATH = PROJECT_DIR / "outputs" / "output.wav"
if os.name == "nt":
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

TEXT_INPUT = "انا سبت الشغل و راجع دلوقتي علي طول."
AUDIO_PROMPT = PROJECT_DIR / "female_voice.wav"

logger.info("Start Cell 3")
print(f"Logging to: {LOG_FILE}")

try:
    script = f"""
from pathlib import Path
import app

text_input = {TEXT_INPUT!r}
audio_prompt = Path({str(AUDIO_PROMPT)!r})
output_path = Path({str(OUTPUT_PATH)!r})

print('text_input:', text_input)
print('audio_prompt:', audio_prompt)
print('output_path:', output_path)
print('default text (ar):', app.default_text_for_ui('ar'))
print('default prompt (ar):', app.default_audio_for_ui('ar'))
"""
    run_cmd([str(VENV_PYTHON), "-c", script], cwd=str(PROJECT_DIR))
except Exception:
    logger.exception("Cell 3 failed")
    raise

logger.info("End Cell 3")


In [ ]:
# Cell 4: Generate audio inside the virtualenv
from pathlib import Path
import logging
import os
import subprocess

CELL_NAME = "cell_04_generate"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
OUTPUT_PATH = PROJECT_DIR / "outputs" / "output.wav"
if os.name == "nt":
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

TEXT_INPUT = "انا سبت الشغل و راجع دلوقتي علي طول."
AUDIO_PROMPT = PROJECT_DIR / "female_voice.wav"

logger.info("Start Cell 4")
print(f"Logging to: {LOG_FILE}")

try:
    script = f"""
from pathlib import Path
import soundfile as sf
import app

text_input = {TEXT_INPUT!r}
audio_prompt = Path({str(AUDIO_PROMPT)!r})
output_path = Path({str(OUTPUT_PATH)!r})
output_path.parent.mkdir(parents=True, exist_ok=True)

sr, wav = app.generate_tts_audio(
    text_input=text_input,
    audio_prompt_path_input=audio_prompt,
    exaggeration_input=0.5,
    temperature_input=0.8,
    seed_num_input=0,
    cfgw_input=0.5,
 )
sf.write(output_path, wav, sr)
print('Saved:', output_path)
print('Sample rate:', sr)
print('Shape:', getattr(wav, 'shape', None))
"""
    run_cmd([str(VENV_PYTHON), "-c", script], cwd=str(PROJECT_DIR))
except Exception:
    logger.exception("Cell 4 failed")
    raise

logger.info("End Cell 4")


In [ ]:
# Cell 5: Preview the generated audio
from pathlib import Path
import logging
import os

CELL_NAME = "cell_05_preview"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
OUTPUT_PATH = PROJECT_DIR / "outputs" / "output.wav"

logger.info("Start Cell 5")
print(f"Logging to: {LOG_FILE}")

try:
    from IPython.display import Audio, display

    if OUTPUT_PATH.exists():
        display(Audio(str(OUTPUT_PATH), autoplay=False))
        print("Audio preview ready.")
        logger.info("Audio preview displayed")
    else:
        print("Output file not found. Run Cell 4 first.")
        logger.warning("Output file not found")
except Exception:
    logger.exception("Cell 5 failed")
    raise

logger.info("End Cell 5")
